# 05 — Course-Fit Model (Phase 4)
## Player × Course Feature Interactions

**Purpose:** Estimate how each player's performance varies with course characteristics.

**Model:** `CourseFit_{i,c} = γ_c · δ_i`
- γ_c = standardized course features (length, rough, wind, etc.)
- δ_i = player-specific interaction coefficients

**Why feature-based (not course fixed effects)?**
Feature-based generalizes to courses a player has never visited. A player who thrives on long, windy courses is predicted to do well at ANY long, windy course.

**Validation:** Leave-one-course-out cross-validation.


In [ ]:
# --- Setup ---
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from config.settings import Settings
from data.loader import DataLoader
from models.course_fit import CourseFitModel

cfg = Settings()
loader = DataLoader(cfg)
rounds_df = loader.load_rounds()

print(f"Loaded {len(rounds_df):,} rounds")
print(f"Course feature names: {cfg.COURSE_FEATURE_NAMES}")


In [ ]:
# --- Load Course Features ---
# You need a course_features.csv with columns matching cfg.COURSE_FEATURE_NAMES
# If not available yet, this notebook shows how to set up the structure.

try:
    course_df = loader.load_course_features()
    print(f"Course features loaded: {course_df.shape}")
    print(f"Courses: {course_df['course_id'].nunique()}")
    course_df.head()
except FileNotFoundError:
    print("⚠️ course_features.csv not found in DATA_DIR.")
    print("\nTo use this notebook, create a CSV with columns:")
    print(f"  course_id, {', '.join(cfg.COURSE_FEATURE_NAMES)}")
    print("\nSources for course data:")
    print("  • DataGolf course metadata")
    print("  • PGA Tour course guides")
    print("  • Manual research")
    course_df = None


In [ ]:
# --- Prepare Residuals ---
# Residuals = actual SG - model estimate (from baseline or Bayesian)
# For now, use raw SG as proxy (residual = SG - player's overall mean)

if course_df is not None:
    player_means = rounds_df.groupby("player_id")["sg_total"].mean()
    residuals_df = rounds_df[["player_id", "event_id", "sg_total"]].copy()
    
    if "course_id" in rounds_df.columns:
        residuals_df["course_id"] = rounds_df["course_id"]
    elif "event_id" in rounds_df.columns:
        # Approximate: map event_id to course_id via events data
        try:
            events_df = loader.load_events()
            if "course_id" in events_df.columns:
                event_course = events_df.set_index("event_id")["course_id"]
                residuals_df["course_id"] = residuals_df["event_id"].map(event_course)
        except:
            print("Cannot map events to courses. Course-fit requires course_id column.")
    
    residuals_df["residual"] = residuals_df["player_id"].map(player_means)
    residuals_df["residual"] = residuals_df["sg_total"] - residuals_df["residual"]
    
    print(f"Residuals computed: {len(residuals_df):,} rows")
    print(f"Residual stats: mean={residuals_df['residual'].mean():.4f}, std={residuals_df['residual'].std():.3f}")


In [ ]:
# --- Fit Course-Fit Model ---
if course_df is not None and "course_id" in residuals_df.columns:
    cf_model = CourseFitModel(cfg, prior_variance=0.01)
    cf_model.fit(residuals_df.dropna(subset=["course_id", "residual"]), course_df)
    
    print(f"Fitted δ_i for {len(cf_model.player_fits)} players")
    
    # Leave-one-course-out CV
    cv_results = cf_model.leave_one_course_out_cv(
        residuals_df.dropna(subset=["course_id", "residual"]),
        course_df
    )
    if len(cv_results) > 0:
        print(f"\nLOCO-CV Results:")
        print(cv_results.describe().round(3))
else:
    print("Skipping course-fit — need course_id and course features")


---
## ✅ Course-Fit Analysis Complete

**Decision point:** Does the course-fit term improve predictions?
- If LOCO-CV shows consistent improvement → keep it (Phase 4 adds value)
- If improvement is noisy or negative → drop it (added complexity, no signal)

**Next step:** → **06_simulation.ipynb**
